Logistic Regression Attempt
---

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display
from sklearn.base import clone
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, GroupKFold, GridSearchCV
from sklearn.metrics import log_loss, confusion_matrix, accuracy_score

px.defaults.template = "plotly_white"
px.defaults.color_discrete_sequence = px.colors.qualitative.Dark24

In [2]:
df = pd.read_csv('../Loan_default.csv')
df

,LoanID,Age,Income,LoanAmount,CreditScore,MonthsEmployed,NumCreditLines,InterestRate,LoanTerm,DTIRatio,Education,EmploymentType,MaritalStatus,HasMortgage,HasDependents,LoanPurpose,HasCoSigner,Default
0,I38PQUQS96,56,85994,50587,520,80,4,15.23,36,0.44,Bachelor's,Full-time,Divorced,Yes,Yes,Other,Yes,0
1,HPSK72WA7R,69,50432,124440,458,15,1,4.81,60,0.68,Master's,Full-time,Married,No,No,Other,Yes,0
2,C1OZ6DPJ8Y,46,84208,129188,451,26,3,21.17,24,0.31,Master's,Unemployed,Divorced,Yes,Yes,Auto,No,1
3,V2KKSFM3UN,32,31713,44799,743,0,3,7.07,24,0.23,High School,Full-time,Married,No,No,Business,No,0
4,EY08JDHTZP,60,20437,9139,633,8,4,6.51,48,0.73,Bachelor's,Unemployed,Divorced,No,Yes,Auto,No,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
255342,8C6S86ESGC,19,37979,210682,541,109,4,14.11,12,0.85,Bachelor's,Full-time,Married,No,No,Other,No,0
255343,98R4KDHNND,32,51953,189899,511,14,2,11.55,24,0.21,High School,Part-time,Divorced,No,No,Home,No,1
255344,XQK1UUUNGP,56,84820,208294,597,70,3,5.29,60,0.50,High School,Self-employed,Married,Yes,Yes,Auto,Yes,0
255345,JAO28CPL4H,42,85109,60575,809,40,1,20.90,48,0.44,High School,Part-time,Single,Yes,Yes,Other,No,0


In [3]:
df.columns

Index(['LoanID', 'Age', 'Income', 'LoanAmount', 'CreditScore',
       'MonthsEmployed', 'NumCreditLines', 'InterestRate', 'LoanTerm',
       'DTIRatio', 'Education', 'EmploymentType', 'MaritalStatus',
       'HasMortgage', 'HasDependents', 'LoanPurpose', 'HasCoSigner',
       'Default'],
      dtype='str')

In [4]:
df.dtypes

LoanID                str
Age                 int64
Income              int64
LoanAmount          int64
CreditScore         int64
MonthsEmployed      int64
NumCreditLines      int64
InterestRate      float64
LoanTerm            int64
DTIRatio          float64
Education             str
EmploymentType        str
MaritalStatus         str
HasMortgage           str
HasDependents         str
LoanPurpose           str
HasCoSigner           str
Default             int64
dtype: object

In [5]:
df = df.drop('LoanID', axis=1)
df

,Age,Income,LoanAmount,CreditScore,MonthsEmployed,NumCreditLines,InterestRate,LoanTerm,DTIRatio,Education,EmploymentType,MaritalStatus,HasMortgage,HasDependents,LoanPurpose,HasCoSigner,Default
0,56,85994,50587,520,80,4,15.23,36,0.44,Bachelor's,Full-time,Divorced,Yes,Yes,Other,Yes,0
1,69,50432,124440,458,15,1,4.81,60,0.68,Master's,Full-time,Married,No,No,Other,Yes,0
2,46,84208,129188,451,26,3,21.17,24,0.31,Master's,Unemployed,Divorced,Yes,Yes,Auto,No,1
3,32,31713,44799,743,0,3,7.07,24,0.23,High School,Full-time,Married,No,No,Business,No,0
4,60,20437,9139,633,8,4,6.51,48,0.73,Bachelor's,Unemployed,Divorced,No,Yes,Auto,No,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
255342,19,37979,210682,541,109,4,14.11,12,0.85,Bachelor's,Full-time,Married,No,No,Other,No,0
255343,32,51953,189899,511,14,2,11.55,24,0.21,High School,Part-time,Divorced,No,No,Home,No,1
255344,56,84820,208294,597,70,3,5.29,60,0.50,High School,Self-employed,Married,Yes,Yes,Auto,Yes,0
255345,42,85109,60575,809,40,1,20.90,48,0.44,High School,Part-time,Single,Yes,Yes,Other,No,0


In [6]:
y = df['Default']
x = df[['Age', 'Income', 'LoanAmount', 'CreditScore',
       'MonthsEmployed', 'NumCreditLines', 'InterestRate', 'LoanTerm',
       'DTIRatio', 'Education', 'EmploymentType', 'MaritalStatus',
       'HasMortgage', 'HasDependents', 'LoanPurpose', 'HasCoSigner']]
cats = ['Education',     
        'EmploymentType',
        'MaritalStatus',
        'HasMortgage',  
        'HasDependents',
        'LoanPurpose',
        'HasCoSigner']
nums = ['Age',         
        'Income',  
        'LoanAmount',    
        'CreditScore',   
        'MonthsEmployed',
        'NumCreditLines',
        'InterestRate',  
        'LoanTerm',
        'DTIRatio'
        ]

In [7]:
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=6021, stratify=y
)

In [8]:
preprocess = ColumnTransformer(transformers=[
    ("encoder", OneHotEncoder(drop='first'), cats),
    ("numeric", StandardScaler(), nums)
])

In [9]:
logreg=LogisticRegression(max_iter=1000, random_state=123)
pipe = Pipeline(steps=[("preprocess", preprocess), ("model", logreg)])
pipe.fit(x_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocess', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](16,)","['Age','Income','LoanAmount',...,'HasDependents','LoanPurpose', 'HasCoSigner']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,16
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('encoder', ...), ('numeric', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthr

In [12]:
p_train = pipe.predict_proba(x_train)[:,1]
p_test = pipe.predict_proba(x_test)[:,1]

log_odds_train = pipe.decision_function(x_train)
log_odds_test = pipe.decision_function(x_test)
threshold = 0.5

In [14]:
print("Number of features:", len(x_train.columns))
print("Number of coefficients:", len(logreg.coef_[0]))
print("Model expected features:", logreg.n_features_in_)

Number of features: 16
Number of coefficients: 24
Model expected features: 24


In [15]:
print("Intercept:", logreg.intercept_[0])

coef = pd.DataFrame({
    "Feature": pipe.named_steps["preprocess"].get_feature_names_out(), #x_train.columns
    "Coefficient": logreg.coef_[0],
    "Odds Ratio": np.exp(logreg.coef_[0])
})

coef

Intercept: -2.088967791645449


,Feature,Coefficient,Odds Ratio
0,encoder__Education_High School,0.090011,1.094187
1,encoder__Education_Master's,-0.108913,0.896808
2,encoder__Education_PhD,-0.164251,0.848529
3,encoder__EmploymentType_Part-time,0.278911,1.321689
4,encoder__EmploymentType_Self-employed,0.226532,1.254243
5,encoder__EmploymentType_Unemployed,0.440726,1.553835
6,encoder__MaritalStatus_Married,-0.237450,0.788636
7,encoder__MaritalStatus_Single,-0.078900,0.924133
8,encoder__HasMortgage_Yes,-0.163471,0.849191
9,encoder__HasDependents_Yes,-0.236592,0.789314


In [17]:
y_pred = (p_test >= threshold).astype(int)

cm = confusion_matrix(y_test, y_pred, labels = [1,0])

cm_df = pd.DataFrame(
    cm,
    index = ["Actual Default", "Actual Non-Default"],
    columns = ["Predicted Default", 'Predicted Non-Default']
)

cm_df

,Predicted Default,Predicted Non-Default
Actual Default,208,5723
Actual Non-Default,127,45012


In [18]:
fig = px.imshow(
    cm_df,
    text_auto=True,
    color_continuous_scale="Blues",
    labels=dict(x="Predicted", y="Actual", color="Count"),
    title="Loan Default Confusion Matrix"
)

fig.update_layout(
    xaxis_title="Predicted Outcome",
    yaxis_title="Actual Outcome"
)

fig.show()

In [20]:
results_test = pd.DataFrame({
    "Set": "Test", "Actual_default": y_test,
    "Predicted_prob_default": p_test,
    "Log_odds": log_odds_test,
    "Predicted_label": (p_test >= threshold).astype(int)
})
results_test.head().round(3)

,Set,Actual_default,Predicted_prob_default,Log_odds,Predicted_label
59805,Test,0,0.040,-3.172,0
13879,Test,0,0.207,-1.345,0
69152,Test,0,0.178,-1.530,0
221284,Test,1,0.499,-0.003,0
45192,Test,0,0.075,-2.507,0


In [21]:
pd.DataFrame({
    "Set": ["Train", "Test"],
    "Log loss": [log_loss(y_train, p_train), log_loss(y_test, p_test)],
    "Accuracy": [accuracy_score(y_train, p_train >= threshold),
                 accuracy_score(y_test, p_test >= threshold)]
})

,Set,Log loss,Accuracy
0,Train,0.316156,0.885073
1,Test,0.314763,0.885451
